# Hola AI — Deal Engine
**Run each cell top to bottom. Takes about 2 minutes to set up.**

This notebook finds government auction properties, underwrites them with Claude AI, and manages your deal pipeline — all from your browser.

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────
!pip install anthropic supabase httpx twilio python-dotenv -q
print('✓ Ready')

In [ ]:
# ── CELL 2: Enter your API keys ───────────────────────────────
# Get yours at console.anthropic.com (free $5 credit on signup)
ANTHROPIC_API_KEY = 'sk-ant-...'   # REQUIRED — paste here

# Get yours at supabase.com → new project → Settings → API
SUPABASE_URL = 'https://xxxx.supabase.co'  # REQUIRED
SUPABASE_KEY = 'your-service-role-key'      # REQUIRED

# Twilio — only needed to send SMS (optional for now)
TWILIO_SID    = ''
TWILIO_TOKEN  = ''
TWILIO_NUMBER = ''

print('✓ Keys saved (not sent anywhere — only used in this notebook)')

In [ ]:
# ── CELL 3: Set up Supabase schema ────────────────────────────
# Run this ONCE to create your tables.
# After that, skip to Cell 4.
from supabase import create_client

sb = create_client(SUPABASE_URL, SUPABASE_KEY)

schema = """
CREATE EXTENSION IF NOT EXISTS "uuid-ossp";

CREATE TABLE IF NOT EXISTS auction_properties (
    id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
    created_at TIMESTAMPTZ DEFAULT NOW(),
    address TEXT NOT NULL,
    city TEXT, state TEXT, zip TEXT, county TEXT,
    source TEXT NOT NULL,
    source_url TEXT, auction_date DATE, listing_id TEXT,
    opening_bid NUMERIC(12,2), current_bid NUMERIC(12,2),
    estimated_arv NUMERIC(12,2), estimated_repairs NUMERIC(12,2), mao NUMERIC(12,2),
    bedrooms INT, bathrooms NUMERIC(4,1), sqft INT, year_built INT,
    property_type TEXT, condition TEXT,
    ai_status TEXT DEFAULT 'PENDING',
    ai_grade TEXT, ai_notes TEXT, sms_draft TEXT,
    status TEXT DEFAULT 'NEW',
    assigned_buyer_id UUID
);

CREATE TABLE IF NOT EXISTS cash_buyers (
    id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
    created_at TIMESTAMPTZ DEFAULT NOW(),
    name TEXT NOT NULL, company TEXT, phone TEXT, email TEXT,
    state TEXT, city TEXT,
    max_purchase_price NUMERIC(12,2),
    preferred_states TEXT[], preferred_property_types TEXT[],
    buys_as_is BOOLEAN DEFAULT TRUE, proof_of_funds BOOLEAN DEFAULT FALSE,
    last_contacted TIMESTAMPTZ, response_rate NUMERIC(4,1),
    deals_closed INT DEFAULT 0,
    opt_in BOOLEAN DEFAULT FALSE, opt_in_date TIMESTAMPTZ,
    opt_out BOOLEAN DEFAULT FALSE,
    score INT DEFAULT 0, status TEXT DEFAULT 'NEW',
    source TEXT
);

CREATE TABLE IF NOT EXISTS active_deals (
    id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
    created_at TIMESTAMPTZ DEFAULT NOW(),
    property_id UUID, buyer_id UUID,
    purchase_price NUMERIC(12,2), auction_date DATE, closing_date_buy DATE,
    assignment_fee NUMERIC(12,2), sale_price NUMERIC(12,2), closing_date_sell DATE,
    title_cost NUMERIC(10,2), holding_cost NUMERIC(10,2), misc_cost NUMERIC(10,2),
    status TEXT DEFAULT 'OPEN', notes TEXT
);

CREATE TABLE IF NOT EXISTS closed_deals (
    id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
    closed_at TIMESTAMPTZ DEFAULT NOW(),
    deal_id UUID, property_address TEXT,
    purchase_price NUMERIC(12,2), sale_price NUMERIC(12,2),
    net_profit NUMERIC(12,2), days_to_close INT, source TEXT
);

CREATE TABLE IF NOT EXISTS outreach_log (
    id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
    sent_at TIMESTAMPTZ DEFAULT NOW(),
    buyer_id UUID, property_id UUID,
    channel TEXT, message TEXT,
    status TEXT, reply_text TEXT, replied_at TIMESTAMPTZ
);
"""

# Execute via Supabase SQL (requires service_role key)
try:
    sb.rpc('exec_sql', {'sql': schema}).execute()
    print('✓ Schema created')
except Exception as e:
    print(f'Note: {e}')
    print('If you see a permission error, paste database/supabase_schema.sql into')
    print('Supabase dashboard → SQL Editor → Run')

In [ ]:
# ── CELL 4: AI Underwriter ────────────────────────────────────
# Paste any auction property details and get an instant deal decision.
import anthropic, json

SYSTEM_PROMPT = open('prompts/underwriter_system_prompt.txt').read() \
    if __import__('os').path.exists('prompts/underwriter_system_prompt.txt') \
    else """
You are a real estate investment underwriter. Given a property, calculate:
1. ARV (After Repair Value) from the comps provided
2. Repair estimate based on year built and condition
3. MAO = (ARV x 0.70) - Repairs
4. Net profit = MAO - opening_bid - 6000 (costs)
Approve if net profit > $10,000. Reject otherwise.
Return ONLY valid JSON: {status, ai_grade, estimated_arv, estimated_repairs,
mao, recommended_bid, net_profit_estimate, sms_draft, reject_reason}
"""

def underwrite(property_data: dict) -> dict:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    msg = client.messages.create(
        model='claude-3-5-sonnet-20241022',
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': f'Underwrite this property:\n{json.dumps(property_data, indent=2)}'}]
    )
    raw = msg.content[0].text.strip().strip('```json').strip('```').strip()
    return json.loads(raw)

print('✓ Underwriter ready')

In [ ]:
# ── CELL 5: Underwrite a real deal ───────────────────────────
# Edit these values with any auction property you find.
# Find properties at: hudhomestore.gov | homepath.com | auction.com | govease.com

property = {
    'address':       '1423 Oak Street',
    'city':          'Memphis',
    'state':         'TN',
    'year_built':    1978,
    'sqft':          1200,
    'bedrooms':      3,
    'bathrooms':     1.5,
    'condition':     'FAIR',    # EXCELLENT | GOOD | FAIR | POOR
    'property_type': 'SFR',
    'opening_bid':   9000,      # What the auction starts at
    'source':        'TAX_SALE',
    'nearby_comps': [
        {'address': '1401 Oak St', 'sold_price': 88000, 'sqft': 1150, 'sold_date': '2025-03-10'},
        {'address': '1445 Elm St', 'sold_price': 92000, 'sqft': 1280, 'sold_date': '2025-02-20'},
    ]
}

result = underwrite(property)

# Pretty print the decision
status = result.get('status')
grade  = result.get('ai_grade', '')
profit = result.get('net_profit_estimate', 0)
arv    = result.get('estimated_arv', 0)
mao    = result.get('mao', 0)
bid    = result.get('recommended_bid', 0)
sms    = result.get('sms_draft', '')

print(f"""\n{'='*50}
DEAL DECISION: {status} {f'[{grade}]' if grade else ''}
{'='*50}
ARV:              ${arv:>10,.0f}
Est. Repairs:     ${result.get('estimated_repairs',0):>10,.0f}
MAO (your ceil):  ${mao:>10,.0f}
Recommended bid:  ${bid:>10,.0f}
Net Profit Est:   ${profit:>10,.0f}
{'='*50}
SMS Draft:
{sms}
{'='*50}"""
)

if result.get('reject_reason'):
    print(f'Reject reason: {result["reject_reason"]}')

In [ ]:
# ── CELL 6: Save approved deal to Supabase ────────────────────
# Only run this if the decision above was APPROVE

if result.get('status') == 'APPROVE':
    from supabase import create_client
    sb = create_client(SUPABASE_URL, SUPABASE_KEY)

    row = {
        **{k: property[k] for k in ['address','city','state','zip','source',
                                      'opening_bid','bedrooms','bathrooms',
                                      'sqft','year_built','property_type','condition']
           if k in property},
        'estimated_arv':     result.get('estimated_arv'),
        'estimated_repairs': result.get('estimated_repairs'),
        'mao':               result.get('mao'),
        'ai_status':         'APPROVE',
        'ai_grade':          result.get('ai_grade'),
        'sms_draft':         result.get('sms_draft'),
        'status':            'APPROVED',
    }

    resp = sb.table('auction_properties').insert(row).execute()
    deal_id = resp.data[0]['id']
    print(f'✓ Deal saved to Supabase: {deal_id}')
    print(f'  Bid up to ${result["mao"]:,.0f} at auction.')
else:
    print('Deal rejected — not saved. Try a different property.')

In [ ]:
# ── CELL 7: Add a cash buyer (optional) ──────────────────────
# Add buyers who want deals texted to them.
# They MUST have opted in (e.g. filled out your funnel page).

from supabase import create_client
from datetime import datetime, timezone
sb = create_client(SUPABASE_URL, SUPABASE_KEY)

buyer = {
    'name':                     'Marcus Johnson',
    'company':                  'MJ Capital LLC',
    'phone':                    '+15551234567',   # must be E.164 format
    'email':                    'marcus@mjcap.com',
    'state':                    'TN',
    'preferred_states':         ['TN', 'GA', 'FL'],
    'max_purchase_price':       150000,
    'preferred_property_types': ['SFR'],
    'buys_as_is':               True,
    'opt_in':                   True,             # only True if they signed your form
    'opt_in_date':              datetime.now(timezone.utc).isoformat(),
    'source':                   'MANUAL',
    'status':                   'ACTIVE',
    'score':                    65,
}

sb.table('cash_buyers').insert(buyer).execute()
print(f'✓ Buyer added: {buyer["name"]}')

In [ ]:
# ── CELL 8: Send SMS to buyer (needs Twilio keys in Cell 2) ──
# Only runs if TWILIO_SID is set.

if not TWILIO_SID:
    print('Add Twilio keys in Cell 2 to enable SMS.')
else:
    from twilio.rest import Client
    tw = Client(TWILIO_SID, TWILIO_TOKEN)

    # Send the SMS draft from the underwriter
    to_number = '+15551234567'   # buyer's phone
    message   = result.get('sms_draft', '') + ' Reply STOP to opt out.'

    msg = tw.messages.create(body=message, from_=TWILIO_NUMBER, to=to_number)
    print(f'✓ SMS sent | SID: {msg.sid}')
    print(f'  To: {to_number}')
    print(f'  Message: {message}')

In [ ]:
# ── CELL 9: Close a deal (run when you've assigned to a buyer) 

from supabase import create_client
from datetime import date
sb = create_client(SUPABASE_URL, SUPABASE_KEY)

# Fill these in when you close
PROPERTY_ADDRESS = '1423 Oak Street, Memphis TN'
PURCHASE_PRICE   = 22000    # what you paid at auction
SALE_PRICE       = 34000    # what your buyer paid you
ASSIGNMENT_FEE   = 12000    # your profit before costs
MISC_COSTS       = 1500     # title, holding, etc.

net = ASSIGNMENT_FEE - MISC_COSTS

sb.table('closed_deals').insert({
    'property_address': PROPERTY_ADDRESS,
    'purchase_price':   PURCHASE_PRICE,
    'sale_price':       SALE_PRICE,
    'net_profit':       net,
    'source':           'TAX_SALE',
}).execute()

print(f'✓ Deal closed and archived')
print(f'  Net profit: ${net:,.0f}')

In [ ]:
# ── CELL 10: Weekly P&L dashboard ─────────────────────────────

from supabase import create_client
from datetime import date, timedelta
sb = create_client(SUPABASE_URL, SUPABASE_KEY)

TARGET = 18000
week_ago = (date.today() - timedelta(days=7)).isoformat()

closed = sb.table('closed_deals').select('net_profit').gte('closed_at', week_ago).execute()
pipeline = sb.table('auction_properties').select('id', count='exact').eq('ai_status','APPROVE').execute()
buyers = sb.table('cash_buyers').select('id', count='exact').eq('opt_in', True).execute()

revenue   = sum(d.get('net_profit',0) or 0 for d in (closed.data or []))
deals     = len(closed.data or [])
pct       = min(100, int(revenue/TARGET*100))
bar       = ('█' * (pct//10)).ljust(10,'░')

print(f"""
{'='*45}
  TRANCHI AI — WEEKLY DASHBOARD
{'='*45}
  Deals closed (7 days):   {deals}
  Revenue (7 days):        ${revenue:>10,.0f}
  Target:                  ${TARGET:>10,.0f}
  Gap:                     ${max(0,TARGET-revenue):>10,.0f}
  Progress: [{bar}] {pct}%

  Approved deals ready:    {pipeline.count or 0}
  Opted-in buyers:         {buyers.count or 0}
{'='*45}
""")